# Exploratory Analysis

Quick notebook for poking at evaluation results interactively. Run `scripts/run_evaluation.py` first, then run this notebook against the produced directory.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import load_config, load_ground_truth, load_predictions, load_metadata
from src.evaluator import Evaluator
from src.failure_analysis import FailureMiner, generate_recommendations, per_image_failure_counts
from src.curation import run_curation

ROOT = Path.cwd().parent
config = load_config(ROOT / 'config' / 'default.yaml')
gt = load_ground_truth(ROOT / 'data' / 'sample' / 'ground_truth.json')
preds = load_predictions(ROOT / 'data' / 'sample' / 'predictions.json')
meta = load_metadata(ROOT / 'data' / 'sample' / 'metadata.json')

result = Evaluator(gt, preds, meta, config).run()
print('Overall:', result.overall)
print('mAP50:', result.mAP50)

In [ ]:
result.by_class

In [ ]:
for cond, df in result.by_condition.items():
    print(f'--- {cond} ---')
    print(df)
    print()

In [ ]:
miner = FailureMiner(result, config)
mined = miner.mine()
for k in ['worst_images_by_fn', 'high_confidence_wrong', 'missed_critical']:
    print(k, '->', len(mined[k]))

generate_recommendations(result, mined, config)